# CTI-ATE walkthrough — report → ATT&CK techniques

**Task:** extract the MITRE ATT&CK techniques described in a report.
**Scoring:** set F1 over technique ids, with revoked ids normalised to their current id.
**Cadence:** daily.

In [6]:
import os, sys, json

def _find_root(start):
    d = os.path.abspath(start)
    while d != os.path.dirname(d):
        if os.path.isdir(os.path.join(d, "src", "glokta")):
            return d
        d = os.path.dirname(d)
    raise RuntimeError("could not locate repo root (a dir containing src/glokta)")

ROOT = _find_root(os.getcwd())
sys.path.insert(0, os.path.join(ROOT, "src"))

# Best-effort load of the repo .env so live HF calls have HF_TOKEN; no dotenv dependency.
_envp = os.path.join(ROOT, ".env")
if os.path.exists(_envp):
    for _line in open(_envp):
        _s = _line.strip()
        if _s and not _s.startswith("#") and "=" in _s:
            _k, _v = _s.split("=", 1)
            os.environ.setdefault(_k.strip(), _v.strip())
os.environ.setdefault("TESTING", "1")  # relax settings validators if .env is absent

MODEL = "huggingface/meta-llama/Llama-3.1-8B-Instruct"
LIVE = bool(os.environ.get("HF_TOKEN"))
print("repo root :", ROOT)
print("model     :", MODEL)
print("LIVE calls:", LIVE, "(set HF_TOKEN to enable real inference)")

def run_model(prompt, canned, max_tokens=256):
    """Call the model live if HF_TOKEN is set, else return a canned example response."""
    if LIVE:
        from glokta.infrastructure.cti.inference import complete
        try:
            return complete(MODEL, prompt, max_tokens=max_tokens, timeout=60.0, max_retries=1)
        except Exception as exc:
            print("[live call failed -> canned]", type(exc).__name__, str(exc)[:80])
            return canned
    print("[offline -> canned response]")
    return canned

repo root : /Users/jake/Projects/glokta
model     : huggingface/meta-llama/Llama-3.1-8B-Instruct
LIVE calls: True (set HF_TOKEN to enable real inference)


## 1. Dataflow — advisory → ATE item; reference table normalises revoked ids
`normalise_report` builds the item; the `TECHNIQUE_INDEX` (from the ATT&CK reference table) maps revoked → current.

In [7]:
# A parsed CISA advisory (the shape parse_advisory produces). In production this is built from
# the CISA RSS feed + page text; here we use a clean in-memory example.
ADVISORY = {
    "id": "AA24-100A",
    "published": __import__("datetime").date(2024, 4, 9),
    "actor": "APT29",
    "cves": ["CVE-2024-12345"],
    "techniques": ["T1059", "T1566"],
    "text": ("APT29 conducted an espionage campaign attributed with high confidence to the "
             "Russian SVR, exploiting CVE-2024-12345 and using T1059 and T1566."),
    "sections": {
        "Summary": "APT29 attribution and high-confidence assessment (a CONCLUSION).",
        "Technical Details": "The actors used T1059 and T1566; beaconing was observed to 198.51.100.23.",
        "Indicators of Compromise": "198.51.100.23",
        "MITRE ATT&CK Techniques": "T1059, T1566 (a CONCLUSION/mapping)",
        "Mitigations": "Apply vendor patches and enforce MFA.",
    },
}

# Small in-memory reference indices. In production these come from the reference tables via
# build_technique_index() and build_taa_indices(); inline here to keep the notebook DB-free.
TECHNIQUE_INDEX = {"T1059": "T1059", "T1566": "T1566", "T1064": "T1059"}  # T1064 is revoked -> T1059
ALIAS_INDEX = {"apt29": "APT29", "cozy bear": "APT29", "the dukes": "APT29",
               "apt28": "APT28", "fancy bear": "APT28"}
RELATED_INDEX = {"APT29": {"APT28"}, "APT28": {"APT29"}}

In [8]:
from glokta.infrastructure.cti.connectors.report import normalise_report

ate = next(i for i in normalise_report(ADVISORY) if i.task == "ate")
print("input_text:", ate.input_text)
print("label     :", ate.label)            # {'techniques': ['T1059', 'T1566']}
print("technique index (revoked->current):", TECHNIQUE_INDEX)

input_text: APT29 conducted an espionage campaign attributed with high confidence to the Russian SVR, exploiting CVE-2024-12345 and using T1059 and T1566.
label     : {'techniques': ['T1059', 'T1566']}
technique index (revoked->current): {'T1059': 'T1059', 'T1566': 'T1566', 'T1064': 'T1059'}


## 2. Tasking — prompt + model call

In [9]:
from glokta.infrastructure.cti.prompts import build_prompt, parse_response

prompt = build_prompt("ate", ate.input_text)
print(prompt)
print("-" * 70)
response = run_model(prompt, canned="Techniques observed: T1059, T1566.")
print("model response :", repr(response))
print("parsed techniques:", parse_response("ate", response))

You are a threat-intelligence analyst. Identify the MITRE ATT&CK techniques described in the report excerpt below.

Report excerpt:
APT29 conducted an espionage campaign attributed with high confidence to the Russian SVR, exploiting CVE-2024-12345 and using T1059 and T1566.

Respond with only the ATT&CK technique ids (e.g. T1059), comma-separated.
Answer:
----------------------------------------------------------------------
model response : 'T1059, T1566'
parsed techniques: {'T1566', 'T1059'}


## 3. Scoring — set F1 with revoked normalisation
Pass the `technique_index` via `context` so revoked ids (e.g. T1064 → T1059) still match.

In [10]:
from glokta.infrastructure.cti.evaluator import evaluate_item

ctx = {"technique_index": TECHNIQUE_INDEX}
for name, resp in {
    "exact":          "T1059, T1566",
    "revoked id":     "The actor used T1064 and T1566.",   # T1064 normalises to T1059
    "one missing":    "Only T1059 was seen.",
    "extra wrong":    "T1059, T1566, T1003",
}.items():
    s = evaluate_item("ate", ate.label, resp, ctx)
    print(f"{name:12} score={s.score:.3f} correct={s.correct} pred={s.parsed_output['techniques']}")

exact        score=1.000 correct=True pred=['T1059', 'T1566']
revoked id   score=1.000 correct=True pred=['T1059', 'T1566']
one missing  score=0.667 correct=False pred=['T1059']
extra wrong  score=0.800 correct=False pred=['T1003', 'T1059', 'T1566']


**Takeaway:** ATE is set F1 like RCM, but the ATT&CK reference table lets a *revoked* technique id still count as correct by normalising it to the current id before scoring.